# A whole detector in fast mode, against the clock

Fits every tile of one LSSTCam detector with `FAST_STELLAR_FIELD_CONFIG` (a grid of local PIFF
PSFs from 25 tiles sampled out to the detector's edges, interpolated for every other tile, and a
tile's own PSF refit only where that PSF does not fit its bright stars;
detection 5 sigma -> 3 sigma, a smooth sky mesh between the stars, bright-star terms kept, the
end-of-pass tests off) and
pipe_crowd's sparse solver, using every CPU this notebook server has.

1. setup and build check;
2. how many CPUs this server gives you;
3. one tile, to predict the whole-detector time before committing to it;
4. the whole detector, timed end to end (loading included), then stitched and plotted.

Run with the **LSST** kernel; restart the kernel after the build cell if it builds.

In [ ]:
%matplotlib inline
import os
import subprocess
import sys
import time

# where this repository is checked out (default: the folder above this notebook)
MEAS_CROWD = os.environ.get("MEAS_CROWD_DIR") or os.path.abspath("..")
# MEAS_CROWD = "/sdf/data/rubin/user/abudlong/code/build/meas_crowd_v2"
PKG = os.path.join(MEAS_CROWD, "python")
if not os.path.isdir(os.path.join(PKG, "lsst", "meas", "crowd")):
    raise FileNotFoundError(f"no meas_crowd package under {PKG}: set MEAS_CROWD above")
if PKG not in sys.path:
    sys.path.insert(0, PKG)
import lsst  # noqa: E402
import lsst.meas  # noqa: E402
for mod, sub in ((lsst, "lsst"), (lsst.meas, os.path.join("lsst", "meas"))):
    if os.path.join(PKG, sub) not in list(mod.__path__):   # lsst / lsst.meas were imported before
        mod.__path__.append(os.path.join(PKG, sub))

In [ ]:
import glob

CHECK = """
import importlib.util, sys
spec = importlib.util.spec_from_file_location("_crowdLib", sys.argv[1])
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)
print(hasattr(m, "solveSparseLeastSquares"))
"""

def solverBuilt():
    """True if the compiled extension has the sparse solver (checked in a fresh process)."""
    for so in glob.glob(os.path.join(PKG, "lsst", "meas", "crowd", "_crowdLib*.so")):
        out = subprocess.run([sys.executable, "-c", CHECK, so], capture_output=True, text=True)
        if out.stdout.strip() == "True":
            return True
    return False

if solverBuilt():
    print("C++ extension with the sparse solver: OK")
else:
    print("building the C++ extension ...")
    r = subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=MEAS_CROWD,
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode == 0 and solverBuilt():
        print("\nBUILT -- now restart the kernel (Kernel > Restart) and run from the top.")
    else:
        raise RuntimeError("build failed: see the output above")

In [ ]:
import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
from scipy.spatial import cKDTree  # noqa: E402

import lsst.meas.crowd as crowd  # noqa: E402

print("meas_crowd from:", os.path.dirname(crowd.__file__))
print("C++ kernels:", crowd.HAVE_CXX, "| sparse Eigen solver:", crowd.HAVE_SPARSE_SOLVER)
assert crowd.HAVE_SPARSE_SOLVER, "the sparse solver is not built: run the build cell, restart"

# The notebook server's CPU limit is a quota (e.g. 4 CPUs) on a node whose cores are all visible
# (e.g. 128): without a cap, NumPy's BLAS starts a thread per visible core and the kernel is
# throttled.  Cap the threads at the usable CPUs.
crowd.limitBlasThreads()

## 1. CPUs of this server

In [ ]:
N_JOBS = crowd.availableCpus()            # the CPU quota, not the node's visible cores
try:
    q, p = open("/sys/fs/cgroup/cpu.max").read().split()
    quota = "none" if q == "max" else f"{int(q) / int(p):g}"
except (FileNotFoundError, ValueError):
    quota = "unknown"
print(f"CPU quota: {quota} | CPUs visible: {len(os.sched_getaffinity(0))} -> N_JOBS = {N_JOBS} "
      f"tile workers (one BLAS thread each)")

## 2. Settings and the visit image

`OUT` stays the same between runs so the Gaia catalogue is read from its cache
(`OUT/gaia_cache/`) after the first run; tiles are refitted each time (`overwrite=True`).

In [ ]:
VISIT, DETECTOR = 2025062100433, 144
REPO, COLLECTION = "dp2_prep", "LSSTCam/runs/DRP/DP2"
OUT = "fast_detector"
CFG = crowd.makeConfig(crowd.FAST_STELLAR_FIELD_CONFIG)

T_START = time.perf_counter()             # the clock for the whole run starts here
from lsst.daf.butler import Butler  # noqa: E402
butler = Butler(REPO, collections=COLLECTION)
refs = butler.query_datasets("visit_image", collections="LSSTCam/runs/*", find_first=False,
                             where=f"instrument='LSSTCam' AND visit={VISIT} AND detector={DETECTOR}")
visitImage = butler.get(refs[0])
tLoad = time.perf_counter() - T_START
bb = visitImage.getBBox()
nTiles = len(crowd.tileSpecs(bb.getWidth(), bb.getHeight(), 400, bb.getMinX(), bb.getMinY(), None))
print(f"visit image loaded in {tLoad:.1f} s; {nTiles} tiles")

## 3. One tile first: predict the whole-detector time

Fits the tile containing (1150, 1150) on one CPU, after an untimed warm-up run that loads and caches
the Gaia catalogue.  The detector is cut into 11 x 11 tiles of ~372 px; the whole detector takes
roughly load + ceil(tiles / N_JOBS) x (time per tile), more if some tiles are more crowded.
This test uses the per-tile PSF refit (two passes): the PSF grid needs many tiles.  In the real
run below most tiles start from the grid PSF and need one pass, so it should come out under this
estimate.

In [ ]:
import contextlib, io  # noqa: E401,E402
# warm-up (not timed): loads the Gaia catalogue once and caches it in OUT/gaia_cache/
with contextlib.redirect_stdout(io.StringIO()):
    crowd.fitTiles(VISIT, DETECTOR, config=CFG, outDir=OUT, visitImage=visitImage,
                   region=(1150, 1151, 1150, 1151), overwrite=True, plots=None, repo=REPO,
                   collection=COLLECTION, detectorPsf=False)
t0 = time.perf_counter()
with contextlib.redirect_stdout(io.StringIO()):
    done = crowd.fitTiles(VISIT, DETECTOR, config=CFG, outDir=OUT, visitImage=visitImage,
                          region=(1150, 1151, 1150, 1151), overwrite=True, plots=None, repo=REPO,
                          collection=COLLECTION, detectorPsf=False)   # a 2 px region: exactly one tile
tTile = (time.perf_counter() - t0) / max(len(done), 1)
est = tLoad + np.ceil(nTiles / N_JOBS) * tTile
print(f"one tile: {tTile:.1f} s -> {nTiles} tiles with {N_JOBS} workers: about {est:.0f} s "
      f"({est / 60:.1f} min), if every tile is like this one")

## 4. The whole detector

First the PSF grid: 25 tiles spread over the detector out to its edges are fitted the full way
(their own PIFF PSF, two passes) in `N_JOBS` parallel workers.  Then every other tile, starting
from the interpolation of the nearest local PSFs; each checks it on its bright stars (`PSF chi2`
on its progress line, ~1 = right) and refits its own PSF with a second pass if the chi2 is above
5 (the summary counts them); each writes its tile file and a log in
`OUT/logs/`.  The total printed at the end is loading the visit image (cell 2) + fitting +
stitching; the one-tile test of cell 3 is not counted.

In [ ]:
crowd.solverStats(reset=True, verbose=False)
t0 = time.perf_counter()
paths = crowd.fitTiles(VISIT, DETECTOR, config=CFG, outDir=OUT, visitImage=visitImage,
                       nJobs=N_JOBS, overwrite=True, plots=None, repo=REPO, collection=COLLECTION)
tFit = time.perf_counter() - t0
t0 = time.perf_counter()
mosaic = crowd.stitchTiles(OUT, verbose=False)
tStitch = time.perf_counter() - t0
tTotal = tLoad + tFit + tStitch                         # cell 3's test run not counted
print(f"\n{len(paths)}/{nTiles} tiles | load {tLoad:.0f} s + fit {tFit:.0f} s + stitch {tStitch:.0f} s "
      f"= {tTotal:.0f} s ({tTotal / 60:.1f} min) | {len(mosaic['catalog'])} sources")

In [ ]:
crowd.plotDetector(mosaic)
plt.show()
crowd.plotDetector(mosaic, view=(1000, 1400, 1000, 1400), binning=1)
plt.show()

### Slowest tiles

Each tile's log ends with its time; the slowest set the total when there are more tiles than
workers.

In [ ]:
import glob, re  # noqa: E401,E402
times = []
for log in glob.glob(os.path.join(OUT, "logs", "*.log")):
    m = re.findall(r"wall clock: ([0-9.]+)s", open(log).read())
    if m:
        times.append((float(m[-1]), os.path.basename(log)[:-4]))
times.sort(reverse=True)
if times:
    print(f"per tile: median {np.median([t for t, _ in times]):.1f} s, max {times[0][0]:.1f} s")
    for t, name in times[:8]:
        print(f"  {name}: {t:.1f} s")

### Does the PSF error change across a tile?  (quadrant check)

Reads the tile files (no refit).  Per tile, stars with model peak S/N 15-100 are stacked as
residual / model peak, per quadrant of the interior and over the whole tile.  *chi2 vs 0* ~1 means
the PSF fits; *variation chi2* ~1 means the error pattern is the same in every quadrant (letting
the PSF vary within a tile would not help), >> 1 that it changes across the tile (it would).  A
PSF varying across a tile can give a whole-tile chi2 near 1 (the halves' errors cancel) and only
show up here.  Set `QC_TILES` to the tiles to look at.

In [ ]:
# Quadrant check: does the PSF error change across a tile?  (reads the tile files; no refit)
#
# For each tile: stars with model peak S/N 15-100 (brighter ones carry their own correction
# terms, which hide PSF errors) are cut out of the residual, each box's edge level subtracted,
# divided by the star's model peak, and stacked per quadrant of the tile interior and over all.
#   chi2 vs 0       : mean over the core of (stack / its error)^2  -- ~1 = PSF right
#   variation chi2  : mean over the quadrants' cores of ((quadrant - all) / error)^2
#                     ~1 = the same pattern everywhere (varying the PSF within the tile will not
#                     help); >> 1 = the PSF error changes across the tile
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

QC_DIR = OUT if "OUT" in globals() else "fast_detector"
QC_TILES = ["x0000_y0000", "x3724_y0000", "x0000_y3640", "x3724_y3640", "x1862_y1820"]
QC_HALF, QC_CORE = 8, 4.0
QC_SNR = (15.0, 100.0)


def quadrantCheck(path, half=QC_HALF, core=QC_CORE, snr=QC_SNR, minStars=8):
    with fits.open(path) as hl:
        h = hl["DATA"].header
        data = hl["DATA"].data.astype(float)
        model = hl["MODEL"].data.astype(float)
        var = hl["VARIANCE"].data.astype(float)
        bad = hl["BAD"].data.astype(bool) | ~np.isfinite(data) | ~(var > 0)
        cat = hl["CATALOG"].data
    X0, Y0 = h["X0"], h["Y0"]
    ix0, ix1, iy0, iy1 = h["IX0"], h["IX1"], h["IY0"], h["IY1"]
    xm, ym = 0.5 * (ix0 + ix1), 0.5 * (iy0 + iy1)
    resid = data - model
    ny, nx = data.shape
    yy, xx = np.mgrid[-half:half + 1, -half:half + 1]
    rr = np.hypot(xx, yy)
    outer, inCore = rr > half - 1.5, rr <= core
    groups = {q: [] for q in range(4)}
    for x, y in zip(np.asarray(cat["x"], float), np.asarray(cat["y"], float)):
        if not (ix0 <= x <= ix1 and iy0 <= y <= iy1):
            continue
        cx, cy = int(round(x)) - X0, int(round(y)) - Y0
        if cx - half < 0 or cy - half < 0 or cx + half >= nx or cy + half >= ny:
            continue
        sl = (slice(cy - half, cy + half + 1), slice(cx - half, cx + half + 1))
        if bad[sl].any():
            continue
        pk = model[cy - 1:cy + 2, cx - 1:cx + 2].max()
        s = pk / np.sqrt(var[cy, cx])
        if not (snr[0] <= s < snr[1]) or pk <= 0:
            continue
        r = resid[sl]
        q = (1 if x >= xm else 0) + (2 if y >= ym else 0)        # 0 BL, 1 BR, 2 TL, 3 TR
        groups[q].append((r - np.median(r[outer])) / pk)

    def stack(st):
        st = np.array(st)
        return st.mean(axis=0), st.std(axis=0, ddof=1) / np.sqrt(len(st)), len(st)

    allSt = [s for g in groups.values() for s in g]
    if len(allSt) < 4 * minStars:
        return None
    mAll, eAll, nAll = stack(allSt)
    ok = inCore & (eAll > 0)
    out = dict(nAll=nAll, chi2All=float(np.mean((mAll[ok] / eAll[ok]) ** 2)), mAll=mAll, quads={})
    var_ = []
    for q, g in groups.items():
        if len(g) < minStars:
            out["quads"][q] = None
            continue
        m, e, n = stack(g)
        okq = inCore & (e > 0)
        out["quads"][q] = dict(n=n, m=m, chi2=float(np.mean((m[okq] / e[okq]) ** 2)))
        var_.append(np.mean(((m - mAll)[okq] / e[okq]) ** 2))
    out["chi2Var"] = float(np.mean(var_)) if len(var_) >= 2 else float("nan")
    return out


names = ["BL", "BR", "TL", "TR"]
results = {}
for t in QC_TILES:
    files = glob.glob(os.path.join(QC_DIR, f"tile_{t}*.fits"))
    if not files:
        print(f"{t}: no tile file in {QC_DIR}")
        continue
    r = quadrantCheck(files[0])
    if r is None:
        print(f"{t}: too few usable stars")
        continue
    results[t] = r
    qs = "  ".join(f"{names[q]} {d['chi2']:5.1f} ({d['n']})" if d else f"{names[q]}   -  "
                   for q, d in r["quads"].items())
    print(f"{t}: {r['nAll']} stars | chi2 vs 0: all {r['chi2All']:5.1f} | {qs} | "
          f"variation chi2 {r['chi2Var']:5.1f}")

if results:
    fig, axes = plt.subplots(len(results), 5, figsize=(14, 2.9 * len(results)), squeeze=False)
    for row, (t, r) in enumerate(results.items()):
        lim = np.nanmax(np.abs(r["mAll"])) * 1.2
        panels = [(r["mAll"], f"{t}: all ({r['nAll']})")] + \
                 [(d["m"] if d else np.full_like(r["mAll"], np.nan),
                   f"{names[q]} ({d['n'] if d else 0}), chi2 {d['chi2']:.1f}" if d else names[q])
                  for q, d in r["quads"].items()]
        for ax, (img, title) in zip(axes[row], panels):
            ax.imshow(img, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
            ax.set_title(title, fontsize=8.5)
            ax.set_xticks([]), ax.set_yticks([])
    fig.suptitle("residual / model peak, stacked per tile quadrant (same colour scale per row)",
                 fontsize=11)
    fig.tight_layout()
    plt.show()
print("\nvariation chi2 ~1: the same error pattern in every quadrant (a PSF varying within the "
      "tile will not help); >> 1: the error changes across the tile (it would).")
